In [1]:
import os   
%pwd
os.chdir("../")
%pwd

'c:\\Users\\danm\\Documents\\DLOps_Project'

In [ ]:

os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/MainakDanOpRes/dlops_energy_forecasting_with_MLflow.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "MainakDanOpRes"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "e381214bfbac472ffa32e83956c6a918999ec211"

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    preprocessor_path: Path   # path to preprocessor.pkl from data_transformation
    target_column: str
    window_size: int
    metric_file_name: Path
    mlflow_uri: str
    all_params: dict
    mlflow_uri: str


In [4]:
from src.dlProject_energy_demand_forcasting.constants import *
from src.dlProject_energy_demand_forcasting.utils.utils import read_yaml, create_directories

In [5]:
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH,
                 schema_filepath=SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        all_params = {
            "LSTM": self.params.LSTM,
            "GRU": self.params.GRU
        }
        create_directories([config.root_dir])

        return ModelEvaluationConfig(
            root_dir=Path(config.root_dir),
            model_path=Path(config.model_path),
            test_data_path=Path(config.test_data_path),
            metric_file_name=Path(config.metric_file_name),
            preprocessor_path=Path(config.preprocessor_path),
            mlflow_uri=config.mlflow_uri,
            all_params=all_params,
            window_size=self.params.data_transformation.window_size,
            target_column=self.schema.target_column.name
        )

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import mlflow
import mlflow.keras
from urllib.parse import urlparse
from tensorflow.keras.models import load_model
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from src.dlProject_energy_demand_forcasting.utils.exception import CustomException
from src.dlProject_energy_demand_forcasting.utils.logger import logging
from src.dlProject_energy_demand_forcasting.utils.utils import load_object
# from src.dlProject_energy_demand_forcasting.entity.config_entity import ModelEvaluationConfig


class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def create_sequences(self, data, window_size):
        """Same windowing logic as model_trainer, applied to the test set."""
        X, y = [], []
        for i in range(len(data) - window_size):
            X.append(data[i : i + window_size])
            y.append(data[i + window_size])
        return np.array(X), np.array(y)

    def inverse_transform_target(self, values, preprocessor, target_col):
        """
        Inverts MinMaxScaler scaling for the target column only.
        Rebuilds a same-width dummy array, inverse-transforms, then slices
        back the target column.
        """
        scaler_step = preprocessor.named_steps["scaler"]
        min_max_scaler = scaler_step.scaler
        columns = list(scaler_step.columns)
        col_idx = columns.index(target_col)

        dummy = np.zeros((len(values), len(columns)))
        dummy[:, col_idx] = values.flatten()
        inv = min_max_scaler.inverse_transform(dummy)
        return inv[:, col_idx]

    def eval_metrics(self, actual, pred):
        rmse = float(np.sqrt(mean_squared_error(actual, pred)))
        mae  = float(mean_absolute_error(actual, pred))
        r2   = float(r2_score(actual, pred))
        mape = float(np.mean(np.abs((actual - pred) / np.clip(np.abs(actual), 1e-8, None))) * 100)
        return rmse, mae, r2, mape

    def initiate_model_evaluation(self):
        try:
            # ── load best model name from trainer artifact ─────────────────────
            best_model_info = load_object(
                file_path=os.path.join("artifacts", "model_trainer", "best_model_info.pkl")
            )
            best_model_name = best_model_info.get("model_name", "Unknown")
            logging.info(f"Best model used for evaluation: {best_model_name}")

            # ── load data ──────────────────────────────────────────────────────
            logging.info("Loading transformed test data for evaluation.")
            test_df = pd.read_csv(self.config.test_data_path, index_col="Datetime")

            target = self.config.target_column
            logging.info(f"Target column: {target}")
            test_array = test_df[target].values.reshape(-1, 1)

            logging.info(f"Creating sequences with window size: {self.config.window_size}")
            X_test, y_test_scaled = self.create_sequences(test_array, self.config.window_size)

            # ── load model & preprocessor ──────────────────────────────────────
            logging.info(f"Loading trained model from {self.config.model_path}")
            model = load_model(self.config.model_path, compile=False)

            logging.info(f"Loading preprocessor from {self.config.preprocessor_path}")
            preprocessor = load_object(file_path=self.config.preprocessor_path)

            # ── predict & inverse transform ────────────────────────────────────
            preds_scaled = model.predict(X_test)

            y_test_actual = self.inverse_transform_target(y_test_scaled, preprocessor, target)
            preds_actual  = self.inverse_transform_target(preds_scaled,  preprocessor, target)

            # ── metrics ────────────────────────────────────────────────────────
            rmse,   mae,   r2,   mape   = self.eval_metrics(y_test_actual,          preds_actual)
            rmse_s, mae_s, r2_s, mape_s = self.eval_metrics(y_test_scaled.flatten(), preds_scaled.flatten())

            # numeric-only dict for MLflow (strings go in params/tags)
            numeric_metrics = {
                "rmse": rmse, "mae": mae, "r2_score": r2, "mape": mape,
                "rmse_scaled": rmse_s, "mae_scaled": mae_s,
                "r2_score_scaled": r2_s, "mape_scaled": mape_s,
            }

            # full dict (with model name) for the JSON report
            full_metrics = {"model_name": best_model_name, **numeric_metrics}

            logging.info(f"Evaluation metrics (real units): {full_metrics}")

            # ── save metrics JSON ──────────────────────────────────────────────
            os.makedirs(self.config.root_dir, exist_ok=True)
            with open(self.config.metric_file_name, "w") as f:
                json.dump(full_metrics, f, indent=4)

            # ── MLflow logging ─────────────────────────────────────────────────
            mlflow.set_registry_uri(self.config.mlflow_uri)
            tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

            with mlflow.start_run(run_name=best_model_name):
                # model name as tag and param (both visible in MLflow UI)
                mlflow.set_tag("model_name", best_model_name)
                mlflow.log_params({
                    "best_model":     best_model_name,
                    "window_size":    self.config.window_size,
                    "target_column":  target,
                })
                mlflow.log_metrics(numeric_metrics)   # ← numeric only, no strings

                if tracking_url_type_store != "file":
                    mlflow.keras.log_model(
                        model, "model",
                        registered_model_name=f"EnergyDemandForecast_{best_model_name}"
                    )
                else:
                    mlflow.keras.log_model(model, "model", 
                                           registered_model_name=f"EnergyDemandForecast_{best_model_name}")

            logging.info("Model evaluation completed and logged to MLflow.")
            return full_metrics

        except Exception as e:
            raise CustomException(e, sys)

c:\Users\danm\Documents\DLOps_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
try:
    config = ConfigurationManager()
    print("schema keys   :", list(config.schema.keys()))
    print("target column :", config.schema.target_column.name)
    print("config keys   :", list(config.config.keys()))
    print("mlflow_uri    :", config.config.model_evaluation.mlflow_uri)
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.initiate_model_evaluation()
except Exception as e:
    raise CustomException(e, sys)


[ 2026-07-01 11:10:25,882 ] 31 root - INFO - yaml file: config\config.yaml loaded successfully
[ 2026-07-01 11:10:25,889 ] 31 root - INFO - yaml file: params.yaml loaded successfully
[ 2026-07-01 11:10:25,891 ] 31 root - INFO - yaml file: schema.yaml loaded successfully
[ 2026-07-01 11:10:25,894 ] 52 root - INFO - created directory at: artifacts
schema keys   : ['target_column', 'columns']
target column : Global_active_power
config keys   : ['artifacts_root', 'data_ingestion', 'data_validation', 'data_transformation', 'model_trainer', 'model_evaluation']
mlflow_uri    : https://dagshub.com/MainakDanOpRes/dlops_energy_forecasting_with_MLflow.mlflow
[ 2026-07-01 11:10:25,895 ] 52 root - INFO - created directory at: artifacts/model_evaluation
[ 2026-07-01 11:10:25,896 ] 60 root - INFO - Best model used for evaluation: LSTM
[ 2026-07-01 11:10:25,897 ] 63 root - INFO - Loading transformed test data for evaluation.
[ 2026-07-01 11:10:25,915 ] 67 root - INFO - Target column: Global_active_pow

2026/07/01 11:10:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/01 11:10:29 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\danm\Documents\DLOps_Project
2026/07/01 11:10:32 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.
2026/07/01 11:10:32 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\danm\Documents\DLOps_Project
2026/07/01 11:10:32 INFO mlflow.utils.environment: Detected uv project at c:\Users\danm\Documents\DLOps_Project. Attempting to export requirements via 'uv export'.
2026/07/01 11:10:33 INFO mlflow.utils.uv_utils: Exported 224 dependencies via uv
2026/07/01 11:10:33 INFO mlflow.utils.environment: Successfully exported 224 requirements from uv project. Skipping package capture based inference.
2026/07/01 11:10:34 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`

🏃 View run LSTM at: https://dagshub.com/MainakDanOpRes/dlops_energy_forecasting_with_MLflow.mlflow/#/experiments/0/runs/242286a8dcb9435fbfe421512512dbd4
🧪 View experiment at: https://dagshub.com/MainakDanOpRes/dlops_energy_forecasting_with_MLflow.mlflow/#/experiments/0
[ 2026-07-01 11:10:39,069 ] 129 root - INFO - Model evaluation completed and logged to MLflow.
